In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import joblib
import matplotlib.pyplot as plt

In [3]:
np.random.seed(42)

In [4]:
n = 1200

In [10]:
legit_amount = np.random.normal(loc = 1200, scale =600, size = n).clip(50,10000)

In [11]:
legit_amount

array([2340.71441145, 1163.60351171,  774.95593985, ...,  454.94365471,
       1400.50585035, 1106.84457071])

In [12]:
len(legit_amount)

1200

In [13]:
legit_hours = np.random.randint(7,22, size = n)

In [14]:
legit_hours

array([12, 18, 16, ..., 14, 15,  7])

In [15]:
legit_merchant_risk = np.random.normal(loc =25, scale = 10,size = n).clip(1,70)

In [17]:
legit_merchant_risk

array([28.22974504, 24.38996028, 30.00240469, ..., 30.77978524,
       30.77363525, 23.71340554])

In [18]:
legit_user_avg = np.random.normal(loc = 3000, scale =1500, size =n ).clip(100,150000)

In [19]:
legit_user_avg

array([5110.40211884, 2273.82225812, 4432.90385581, ..., 4590.21572801,
       1659.38097137, 2418.05254664])

In [20]:
fraud_amount = np.random.normal(loc = 4000, scale = 2000, size = n).clip(100,20000)
fraud_hours = np.random.choice(
[0,1,2,3,4,5,6,22,23], size = n, p=[0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.15,0.15]
)
fraud_merchant_risk = np.random.normal(loc = 70, scale = 15, size = n).clip(30,100)
fraud_user_avg = np.random.normal(loc = 800, scale = 400, size = n).clip(50,50000)

In [21]:
x = np.vstack([
    np.column_stack([legit_amount, legit_hours, legit_merchant_risk, legit_user_avg]),
    np.column_stack([fraud_amount, fraud_hours, fraud_merchant_risk, fraud_user_avg])
])

In [22]:
x

array([[2340.71441145,   12.        ,   28.22974504, 5110.40211884],
       [1163.60351171,   18.        ,   24.38996028, 2273.82225812],
       [ 774.95593985,   16.        ,   30.00240469, 4432.90385581],
       ...,
       [3404.40768755,    0.        ,   80.10209194,  516.48871891],
       [5266.45062023,    6.        ,   66.29622509,  472.26336291],
       [3224.74033467,   22.        ,   41.11033272, 1263.38879859]])

In [23]:
y_labels = np.array(["legit"]*n+["fraud"]*n)

In [24]:
y_labels

array(['legit', 'legit', 'legit', ..., 'fraud', 'fraud', 'fraud'],
      dtype='<U5')

In [32]:
df = pd.DataFrame(x, columns = ["amount_inr","txn_hour","merchant_risk_score","user_avg_txn"])

In [33]:
df

,amount_inr,txn_hour,merchant_risk_score,user_avg_txn
0,2340.714411,12.0,28.229745,5110.402119
1,1163.603512,18.0,24.389960,2273.822258
2,774.955940,16.0,30.002405,4432.903856
3,291.771364,20.0,19.663997,2941.885658
4,118.116194,15.0,37.208213,2087.473853
...,...,...,...,...
2395,3745.186255,3.0,58.503004,461.724275
2396,4205.291057,5.0,56.003214,692.840506
2397,3404.407688,0.0,80.102092,516.488719
2398,5266.450620,6.0,66.296225,472.263363


In [34]:
df["labels"] = y_labels

In [35]:
df

,amount_inr,txn_hour,merchant_risk_score,user_avg_txn,labels
0,2340.714411,12.0,28.229745,5110.402119,legit
1,1163.603512,18.0,24.389960,2273.822258,legit
2,774.955940,16.0,30.002405,4432.903856,legit
3,291.771364,20.0,19.663997,2941.885658,legit
4,118.116194,15.0,37.208213,2087.473853,legit
...,...,...,...,...,...
2395,3745.186255,3.0,58.503004,461.724275,fraud
2396,4205.291057,5.0,56.003214,692.840506,fraud
2397,3404.407688,0.0,80.102092,516.488719,fraud
2398,5266.450620,6.0,66.296225,472.263363,fraud


In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400 entries, 0 to 2399
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   amount_inr           2400 non-null   float64
 1   txn_hour             2400 non-null   float64
 2   merchant_risk_score  2400 non-null   float64
 3   user_avg_txn         2400 non-null   float64
 4   labels               2400 non-null   object 
dtypes: float64(4), object(1)
memory usage: 93.9+ KB


In [39]:
x = df.drop("labels", axis = 1)
y = df["labels"]

In [41]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.2, random_state = 42)

In [48]:
base_model = Pipeline(
    steps = [
        
        ("svc", SVC(kernel = "rbf", probability = True))
    ]
)

In [49]:
base_model

Pipeline(steps=[('svc', SVC(probability=True))])

In [50]:
base_model.fit(x_train, y_train)

Pipeline(steps=[('svc', SVC(probability=True))])

In [51]:
base_model.predict(x_test)

array(['fraud', 'fraud', 'legit', 'fraud', 'fraud', 'fraud', 'fraud',
       'fraud', 'fraud', 'legit', 'fraud', 'fraud', 'legit', 'fraud',
       'fraud', 'fraud', 'legit', 'fraud', 'fraud', 'fraud', 'legit',
       'fraud', 'legit', 'fraud', 'fraud', 'legit', 'fraud', 'fraud',
       'fraud', 'fraud', 'fraud', 'legit', 'fraud', 'legit', 'fraud',
       'fraud', 'legit', 'legit', 'legit', 'legit', 'legit', 'fraud',
       'legit', 'legit', 'legit', 'legit', 'fraud', 'fraud', 'fraud',
       'legit', 'legit', 'fraud', 'legit', 'fraud', 'fraud', 'fraud',
       'fraud', 'fraud', 'fraud', 'fraud', 'fraud', 'legit', 'fraud',
       'fraud', 'fraud', 'legit', 'fraud', 'fraud', 'fraud', 'legit',
       'fraud', 'fraud', 'fraud', 'fraud', 'legit', 'legit', 'fraud',
       'legit', 'legit', 'fraud', 'legit', 'legit', 'fraud', 'legit',
       'legit', 'legit', 'legit', 'legit', 'legit', 'legit', 'fraud',
       'legit', 'legit', 'fraud', 'fraud', 'fraud', 'legit', 'fraud',
       'fraud', 'leg

In [52]:
accuracy = base_model.score(x_test, y_test)
print (f"Accuracy of the model is: {accuracy*100:.2f}%")

Accuracy of the model is: 91.88%


In [53]:
grid = GridSearchCV(
    estimator = base_model,
    param_grid = {
        "svc__C":[0.1,1,100],
        "svc__gamma":[0.1,0.1,0.01,0.001],
        "svc__kernel":["linear","rbf"],
        "svc__degree":[2,3,4]
    },
    cv = 5,
    scoring = "accuracy",
    verbose = 2,
    n_jobs = -1
)

In [54]:
grid.fit(x_train, y_train)

Fitting 5 folds for each of 72 candidates, totalling 360 fits
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=0.1, svc__kernel=rbf; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=0.1, svc__kernel=rbf; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=0.1, svc__kernel=rbf; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=0.1, svc__kernel=rbf; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=0.1, svc__kernel=rbf; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=0.1, svc__kernel=rbf; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=0.1, svc__kernel=rbf; total time=   0.4s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=0.1, svc__kernel=rbf; total time=   1.0s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=0.1, svc__kernel=rbf; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=0.1, svc__kernel=rbf; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=0.1, svc__k

GridSearchCV(cv=5, estimator=Pipeline(steps=[('svc', SVC(probability=True))]),
             n_jobs=-1,
             param_grid={'svc__C': [0.1, 1, 100], 'svc__degree': [2, 3, 4],
                         'svc__gamma': [0.1, 0.1, 0.01, 0.001],
                         'svc__kernel': ['linear', 'rbf']},
             scoring='accuracy', verbose=2)

In [55]:
pip install -U scikit-learn

  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/42/e2/ff880f62677a17d035817d543cb0fc8727d01eccbee81c5f7fc733a9d856/scikit_learn-1.9.0-cp311-cp311-macosx_12_0_arm64.whl.metadata
  Obtaining dependency information for joblib>=1.4.0 from https://files.pythonhosted.org/packages/7b/91/984aca2ec129e2757d1e4e3c81c3fcda9d0f85b74670a094cc443d9ee949/joblib-1.5.3-py3-none-any.whl.metadata
  Obtaining dependency information for narwhals>=2.0.1 from https://files.pythonhosted.org/packages/eb/dc/55481808fd70ef1567cf13540ffd4702af3f74b112e35427564b03f79c2d/narwhals-2.25.0-py3-none-any.whl.metadata
  Obtaining dependency information for threadpoolctl>=3.5.0 from https://files.pythonhosted.org/packages/32/d5/f9a850d79b0851d1d4ef6456097579a9005b31fea68726a4ae5f2d82ddd9/threadpoolctl-3.6.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 24.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 

In [57]:
best_model =grid.best_estimator_

In [58]:
grid.best_params_

{'svc__C': 0.1, 'svc__degree': 2, 'svc__gamma': 0.1, 'svc__kernel': 'linear'}

In [59]:
grid.best_score_

0.9916666666666668

In [60]:
best_accuracy = best_model.score(x_test, y_test)
print (f"Accuracy of the best model is: {best_accuracy*100:.2f}%")

Accuracy of the best model is: 98.33%


In [61]:
joblib.dump(best_model, "svc_fraud_model.pkl")

['svc_fraud_model.pkl']